In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f70aa03bf49202b59a323fb8261cbabff7cc4564d72f19f6203f6c064d447acf
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import logging
import re
from datasets import load_dataset
from transformers import BartForConditionalGeneration, BartTokenizer
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader, Dataset
import os
import warnings
warnings.filterwarnings("ignore")

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

nltk.download('punkt', quiet=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Running on: {device}")

# Clean text
def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return text.strip()

# Generate pseudo-summaries
def generate_pseudo_summaries(reviews, max_length=50, min_length=10):
    logger.info("Generating pseudo-summaries for reviews...")
    bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(device)
    bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
    
    summaries = []
    for review in tqdm(reviews, desc="Creating summaries"):
        review = clean_text(review)
        if len(review) < 20:
            summaries.append("")
            continue
        inputs = bart_tokenizer(review, max_length=512, truncation=True, return_tensors='pt').to(device)
        summary_ids = bart_model.generate(
            inputs['input_ids'],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
        summary = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        summaries.append(summary)
    
    return summaries

# Assign pseudo-categories
def assign_pseudo_category(stars):
    return "Positive" if stars >= 4 else "Negative"

# EDA
def perform_eda(dataset, split_name, output_dir="eda_plots"):
    logger.info(f"Running EDA on {split_name} split...")
    df = pd.DataFrame({
        'review': dataset['text'],
        'summary': dataset['summary'],
        'category': dataset['category']
    })
    
    df['review_words'] = df['review'].apply(lambda x: len(word_tokenize(clean_text(x))))
    df['summary_words'] = df['summary'].apply(lambda x: len(word_tokenize(clean_text(x))))
    df['review_sentences'] = df['review'].apply(lambda x: len(sent_tokenize(clean_text(x))))
    df['summary_sentences'] = df['summary'].apply(lambda x: len(sent_tokenize(clean_text(x))))
    
    plt.figure(figsize=(12, 8))
    plt.subplot(2, 2, 1)
    sns.histplot(df['review_words'], bins=50, kde=True)
    plt.title(f'Review Word Counts ({split_name})')
    plt.subplot(2, 2, 2)
    sns.histplot(df['summary_words'], bins=50, kde=True)
    plt.title(f'Summary Word Counts ({split_name})')
    plt.subplot(2, 2, 3)
    sns.histplot(df['review_sentences'], bins=50, kde=True)
    plt.title(f'Review Sentence Counts ({split_name})')
    plt.subplot(2, 2, 4)
    sns.histplot(df['summary_sentences'], bins=50, kde=True)
    plt.title(f'Summary Sentence Counts ({split_name})')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{split_name}_eda.png")
    plt.close()
    
    plt.figure(figsize=(10, 5))
    sns.countplot(x='category', data=df)
    plt.title(f'Category Distribution ({split_name})')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{split_name}_category_distribution.png")
    plt.close()
    
    logger.info(f"{split_name} - Review Words: Mean={df['review_words'].mean():.2f}, Std={df['review_words'].std():.2f}")
    logger.info(f"{split_name} - Summary Words: Mean={df['summary_words'].mean():.2f}, Std={df['summary_words'].std():.2f}")
    logger.info(f"{split_name} - Review Sentences: Mean={df['review_sentences'].mean():.2f}, Std={df['review_sentences'].std():.2f}")
    logger.info(f"{split_name} - Summary Sentences: Mean={df['summary_sentences'].mean():.2f}, Std={df['summary_sentences'].std():.2f}")

# Custom dataset
class ReviewSummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_len=512, max_target_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data['text'])

    def __getitem__(self, idx):
        review = clean_text(self.data['text'][idx])
        summary = clean_text(self.data['summary'][idx])
        
        input_encoding = self.tokenizer(
            review,
            max_length=self.max_input_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        target_encoding = self.tokenizer(
            summary,
            max_length=self.max_target_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }

# Train model
def train_model(model, tokenizer, train_dataset, model_name, epochs=3, batch_size=2):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    logger.info(f"Training {model_name}...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            total_loss += loss.item()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(train_loader)
        logger.info(f"{model_name} - Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")
    
    model_dir = f"summarization_models/{model_name.lower().replace(' ', '_')}"
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    logger.info(f"{model_name} saved to {model_dir}")

# Evaluate model
def evaluate_model(model, tokenizer, val_dataset, model_name):
    model.to(device)
    model.eval()
    val_loader = DataLoader(val_dataset, batch_size=2)
    
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    bleu_scores = []
    
    logger.info(f"Evaluating {model_name}...")
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True
            )
            
            for gen_id, label in zip(generated_ids, labels):
                generated_text = tokenizer.decode(gen_id, skip_special_tokens=True)
                reference_text = tokenizer.decode(label, skip_special_tokens=True)
                
                scores = rouge.score(reference_text, generated_text)
                for key in rouge_scores:
                    rouge_scores[key].append(scores[key].fmeasure)
                
                reference_tokens = [word_tokenize(reference_text)]
                generated_tokens = word_tokenize(generated_text)
                bleu = sentence_bleu(reference_tokens, generated_tokens, weights=(0.25, 0.25, 0.25, 0.25))
                bleu_scores.append(bleu)
    
    avg_rouge = {key: np.mean(scores) for key, scores in rouge_scores.items()}
    avg_bleu = np.mean(bleu_scores)
    
    return {
        'Model': model_name,
        'ROUGE-1': avg_rouge['rouge1'],
        'ROUGE-2': avg_rouge['rouge2'],
        'ROUGE-L': avg_rouge['rougeL'],
        'BLEU': avg_bleu
    }

# Main execution
os.makedirs("eda_plots", exist_ok=True)
os.makedirs("summarization_models", exist_ok=True)

logger.info("Loading Yelp reviews...")
dataset = load_dataset("yelp_review_full", split="train")
num_train_samples = 4000
num_val_samples = 800
train_data = dataset.select(range(num_train_samples))
val_data = dataset.select(range(num_train_samples, num_train_samples + num_val_samples))

train_reviews = [clean_text(r) for r in train_data['text']]
val_reviews = [clean_text(r) for r in val_data['text']]
train_summaries = generate_pseudo_summaries(train_reviews)
val_summaries = generate_pseudo_summaries(val_reviews)

train_categories = [assign_pseudo_category(stars) for stars in train_data['label']]
val_categories = [assign_pseudo_category(stars) for stars in val_data['label']]

train_data = {
    'text': [r for r, s in zip(train_reviews, train_summaries) if s],
    'summary': [s for s in train_summaries if s],
    'category': [c for c, s in zip(train_categories, train_summaries) if s]
}
val_data = {
    'text': [r for r, s in zip(val_reviews, val_summaries) if s],
    'summary': [s for s in val_summaries if s],
    'category': [c for c, s in zip(val_categories, val_summaries) if s]
}

perform_eda(train_data, "train")
perform_eda(val_data, "validation")

model_name = 'BART'
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

train_dataset = ReviewSummarizationDataset(train_data, tokenizer)
val_dataset = ReviewSummarizationDataset(val_data, tokenizer)

train_model(model, tokenizer, train_dataset, model_name, epochs=3, batch_size=2)
result = evaluate_model(model, tokenizer, val_dataset, model_name)
logger.info(f"{model_name} Results: {result}")

results_df = pd.DataFrame([result])
results_df.to_csv('bart_results.csv', index=False)
logger.info("Results saved to bart_results.csv")

# Example: Summarize a new review with the fine-tuned model
def summarize_review(review, model, tokenizer, max_length=128):
    model.eval()
    inputs = tokenizer(review, max_length=512, truncation=True, return_tensors='pt').to(device)
    summary_ids = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Load fine-tuned model
fine_tuned_model = BartForConditionalGeneration.from_pretrained('summarization_models/bart').to(device)
fine_tuned_tokenizer = BartTokenizer.from_pretrained('summarization_models/bart')

# Example review
sample_review = "The food was amazing, and the service was top-notch. Highly recommend this place!"
summary = summarize_review(sample_review, fine_tuned_model, fine_tuned_tokenizer)
logger.info(f"Sample Review: {sample_review}")
logger.info(f"Summary: {summary}")

2025-06-17 09:43:31.785966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750153411.972832      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750153412.030386      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/299M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

...

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Evaluating: 100%|██████████| 398/398 [07:57<00:00,  1.20s/it]
